### Sentiment Analysis 
- Adding Emotion to Robotic Voices (For Speech Impairments)


In [ ]:
import os
import sys
import numpy as np
from transformers import pipeline
import pyttsx3
from IPython.display import Audio
import io
import wave

sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("Model loaded successfully!")

c:\Users\user\OneDrive\Desktop\CODE\CCNLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentiment analysis model...


c:\Users\user\OneDrive\Desktop\CODE\CCNLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1435.22it/s]


Model loaded successfully!


In [2]:
def analyze_sentiment(text):
    """
    Analyze sentiment of input text using DistilBERT model
    More advanced than VADER - uses deep learning transformer
    
    Args:
        text (str): Input text to analyze
    
    Returns:
        dict: Sentiment label and confidence score
    """
    result = sentiment_analyzer(text)[0]
    return {
        "label": result["label"],
        "score": result["score"],
        "text": text
    }

# Test the analyzer
test_text = "I absolutely love this amazing product! It makes me so happy!"
sentiment_result = analyze_sentiment(test_text)
print(f"Input: {sentiment_result['text']}")
print(f"Sentiment: {sentiment_result['label']} (confidence: {sentiment_result['score']:.2%})")

Input: I absolutely love this amazing product! It makes me so happy!
Sentiment: POSITIVE (confidence: 99.99%)


In [3]:
def sentiment_to_voice(text, sentiment_label, confidence):
    """
    Convert sentiment to voice characteristics (speed, volume)
    Adds emotion to robotic voice based on sentiment
    
    Args:
        text (str): Text to speak
        sentiment_label (str): "POSITIVE" or "NEGATIVE"
        confidence (float): Confidence score (0-1)
    
    Returns:
        tuple: (rate, volume) for voice characteristics
    """
    base_rate = 150  # Base speech rate (words per minute)
    base_volume = 0.8
    
    if sentiment_label == "POSITIVE":
        # Positive sentiment: faster, more energetic
        rate = base_rate + int(confidence * 50)  # 150-200 wpm
        volume = min(1.0, base_volume + (confidence * 0.2))
    else:  # NEGATIVE
        # Negative sentiment: slower, more somber
        rate = base_rate - int(confidence * 50)  # 100-150 wpm
        volume = max(0.3, base_volume - (confidence * 0.2))
    
    return rate, volume

# Test voice characteristics
positive_rate, positive_vol = sentiment_to_voice("test", "POSITIVE", 0.95)
negative_rate, negative_vol = sentiment_to_voice("test", "NEGATIVE", 0.90)

print(f"Positive sentiment voice: Rate={positive_rate} wpm, Volume={positive_vol:.1f}")
print(f"Negative sentiment voice: Rate={negative_rate} wpm, Volume={negative_vol:.1f}")

Positive sentiment voice: Rate=197 wpm, Volume=1.0
Negative sentiment voice: Rate=105 wpm, Volume=0.6


In [4]:
def generate_emotional_speech(text, sentiment_label, confidence):
    """
    Generate emotional text-to-speech audio based on sentiment
    Produces actual audio output for the robot to speak
    
    Args:
        text (str): Text to convert to speech
        sentiment_label (str): "POSITIVE" or "NEGATIVE"
        confidence (float): Sentiment confidence score
    
    Returns:
        tuple: (Audio object for Jupyter display, audio_path)
    """
    # Get voice characteristics based on sentiment
    rate, volume = sentiment_to_voice(text, sentiment_label, confidence)
    
    # Initialize text-to-speech engine
    engine = pyttsx3.init()
    engine.setProperty('rate', rate)
    engine.setProperty('volume', volume)
    
    # Get available voices and set to preferred voice
    voices = engine.getProperty('voices')
    if len(voices) > 1:
        # Try to use female voice if available (index 1), otherwise use default
        engine.setProperty('voice', voices[0].id)
    
    # Create audio file in memory
    audio_file = "emotional_speech.wav"
    engine.save_to_file(text, audio_file)
    engine.runAndWait()
    
    # Create IPython Audio object for display
    audio = Audio(audio_file)
    
    return audio, audio_file

# Create a more sophisticated example
print("\\n" + "="*60)
print("SENTIMENT-BASED EMOTIONAL SPEECH SYSTEM")
print("="*60)

\n============================================================
SENTIMENT-BASED EMOTIONAL SPEECH SYSTEM


In [5]:
# Example 1: Positive Sentiment
positive_text = "I am absolutely thrilled and delighted! This is the best day ever!"
print(f"\nExample 1 - Positive Sentiment")
print(f"Text: {positive_text}")

positive_sentiment = analyze_sentiment(positive_text)
print(f"Sentiment: {positive_sentiment['label']} ({positive_sentiment['score']:.1%} confidence)")
print(f"Voice Characteristics: Fast and energetic")
print("\nGenerating emotional speech...")

audio1, file1 = generate_emotional_speech(
    positive_text,
    positive_sentiment['label'],
    positive_sentiment['score']
)
print("✓ Audio generated!")
audio1


Example 1 - Positive Sentiment
Text: I am absolutely thrilled and delighted! This is the best day ever!
Sentiment: POSITIVE (100.0% confidence)
Voice Characteristics: Fast and energetic

Generating emotional speech...
✓ Audio generated!


In [6]:
# Example 2: Negative Sentiment
negative_text = "I am deeply disappointed and upset about this terrible situation."
print(f"\n\nExample 2 - Negative Sentiment")
print(f"Text: {negative_text}")

negative_sentiment = analyze_sentiment(negative_text)
print(f"Sentiment: {negative_sentiment['label']} ({negative_sentiment['score']:.1%} confidence)")
print(f"Voice Characteristics: Slow and somber")
print("\nGenerating emotional speech...")

audio2, file2 = generate_emotional_speech(
    negative_text,
    negative_sentiment['label'],
    negative_sentiment['score']
)
print("✓ Audio generated!")
audio2



Example 2 - Negative Sentiment
Text: I am deeply disappointed and upset about this terrible situation.
Sentiment: NEGATIVE (99.9% confidence)
Voice Characteristics: Slow and somber

Generating emotional speech...
✓ Audio generated!


In [ ]:
def process_user_text(user_input):
    """
    Process user input: analyze sentiment and generate emotional speech
    
    Args:
        user_input (str): Text from user
    
    Returns:
        Audio object and sentiment analysis
    """
    print(f"\nInput Text: '{user_input}'")
    
    # Analyze sentiment
    result = analyze_sentiment(user_input)
    print(f"Sentiment Analysis Result:")
    print(f"  - Label: {result['label']}")
    print(f"  - Confidence: {result['score']:.1%}")
    
    # Generate emotional speech
    emotion = "Happy & Energetic" if result['label'] == "POSITIVE" else "Sad & Slower"
    print(f"  - Emotion Applied: {emotion}")
    
    # Generate audio
    audio, file = generate_emotional_speech(
        user_input,
        result['label'],
        result['score']
    )
    print("\n✓ Emotional speech generated and ready to play!")
    return audio, result

# Test with a neutral/mixed sentiment
test_input = "This product is okay, nothing special but it works."
audio3, result3 = process_user_text(test_input)
audio3



Example 3 - Interactive Sentiment Analysis with Speech Output

Input Text: 'This product is okay, nothing special but it works.'
Sentiment Analysis Result:
  - Label: POSITIVE
  - Confidence: 100.0%
  - Emotion Applied: Happy & Energetic

✓ Emotional speech generated and ready to play!


## System Summary & Features

### Key Components:

1. **Sentiment Analysis Model**: DistilBERT-based classifier
   - Free, open-source transformer model
   - More advanced than VADER (uses deep learning)
   - Provides both sentiment label and confidence score

2. **Emotional Voice Mapping**:
   - **POSITIVE sentiment**: Faster speech rate (150-200 wpm), higher volume
   - **NEGATIVE sentiment**: Slower speech rate (100-150 wpm), lower volume
   - Emotion intensity correlates with model confidence

3. **Text-to-Speech Engine**: pyttsx3
   - Generates actual audio output in .wav format
   - Adjustable rate and volume for prosody
   - Cross-platform support

### Use Case:
Perfect for accessibility applications where robotic voices need emotional expression, especially for users with speech impairments where natural emotional prosody is important for communication quality.

### Model Performance:
- Accuracy on SST-2 benchmark: ~92%
- Latency: Sub-second inference
- Support for various text inputs and domains